# Encoder and Decoder: IMDb Sentiment
Hugging Face: **Sultan0336** | GitHub: **sultanbjan-bit**

Run each cell in order on a Colab GPU. This notebook trains DistilBERT and
DistilGPT2 for the same binary classification task. No results are prefilled.
The decoder has a classification head; it is not used as a chatbot.

## 1. Install training packages
Use a fresh runtime. If Colab asks for a restart, restart and continue from step 2.
Colab supplies PyTorch and its GPU driver; we do not replace them with CPU wheels.


In [ ]:
%pip -q install transformers==4.57.1 datasets==4.2.0 accelerate==1.11.0 scikit-learn==1.7.2 pandas==2.2.3


## 2. Check GPU and create project files
A free GPU is not guaranteed. If none is available, change runtime or try later.

In [ ]:
import torch
assert torch.cuda.is_available(), "Select a GPU runtime before training."
print("GPU:", torch.cuda.get_device_name(0))
print("PyTorch:", torch.__version__)


In [ ]:
from pathlib import Path
import json
SOURCE_FILES = {'project_config.json': '{\n  "hf_username": "Sultan0336",\n  "github_username": "sultanbjan-bit",\n  "seed": 42,\n  "max_length": 128,\n  "epochs": 2,\n  "models": {\n    "Encoder: DistilBERT": {"base": "distilbert/distilbert-base-uncased", "repo": "Sultan0336/distilbert-imdb-sentiment"},\n    "Decoder: DistilGPT2": {"base": "distilbert/distilgpt2", "repo": "Sultan0336/distilgpt2-imdb-sentiment"}\n  }\n}\n', 'text_utils.py': '"""Use the same review cleanup for training and the app."""\nimport html\nimport re\n\n\ndef clean_text(text):\n    text = html.unescape(text)\n    text = re.sub(r"<br\\s*/?>", " ", text, flags=re.IGNORECASE)\n    return " ".join(text.split())\n', 'train_models.py': '"""Training functions used by the Colab notebook. No training starts on import."""\nimport gc\nimport hashlib\nimport json\nimport time\nfrom pathlib import Path\n\nimport numpy as np\nimport pandas as pd\nimport torch\nfrom datasets import Dataset, DatasetDict, load_dataset\nfrom huggingface_hub import HfApi\nfrom sklearn.metrics import accuracy_score, confusion_matrix, precision_recall_fscore_support\nfrom transformers import (AutoModelForSequenceClassification, AutoTokenizer,\n                          DataCollatorWithPadding, Trainer, TrainingArguments, set_seed)\n\nfrom text_utils import clean_text\n\nCONFIG = json.loads(Path("project_config.json").read_text())\nOUTPUT = Path("outputs")\nID2LABEL = {0: "NEGATIVE", 1: "POSITIVE"}\n\n\ndef prepare_data():\n    raw = load_dataset("stanfordnlp/imdb")\n    train = raw["train"].to_pandas()\n    test = raw["test"].to_pandas()\n    for frame in (train, test):\n        frame["text"] = frame["text"].map(clean_text)\n        frame["key"] = frame["text"].str.casefold()\n    train = train.drop_duplicates("key")\n    test = test.drop_duplicates("key")\n    # Remove exact normalized overlaps before any sampling.\n    test = test[~test["key"].isin(train["key"])]\n    pieces = {"train": [], "validation": [], "test": []}\n    for label in (0, 1):\n        pool = train[train.label == label].sample(n=2250, random_state=CONFIG["seed"])\n        pieces["train"].append(pool.iloc[:2000])\n        pieces["validation"].append(pool.iloc[2000:])\n        pieces["test"].append(test[test.label == label].sample(n=500, random_state=CONFIG["seed"]))\n    splits, manifests = {}, {}\n    for split, frames in pieces.items():\n        frame = pd.concat(frames).sample(frac=1, random_state=CONFIG["seed"]).reset_index(drop=True)\n        manifests[split] = [hashlib.sha256(x.encode()).hexdigest() for x in frame["key"]]\n        splits[split] = Dataset.from_pandas(frame[["text", "label"]], preserve_index=False)\n    for a, b in (("train", "validation"), ("train", "test"), ("validation", "test")):\n        assert not set(manifests[a]) & set(manifests[b]), f"Overlap: {a}, {b}"\n    OUTPUT.mkdir(exist_ok=True)\n    (OUTPUT / "split_manifest.json").write_text(json.dumps(manifests, indent=2))\n    return DatasetDict(splits)\n\n\ndef compute_metrics(prediction):\n    predicted = np.argmax(prediction.predictions, axis=-1)\n    precision, recall, f1, _ = precision_recall_fscore_support(\n        prediction.label_ids, predicted, average="macro", zero_division=0)\n    return {"accuracy": accuracy_score(prediction.label_ids, predicted),\n            "macro_precision": precision, "macro_recall": recall, "macro_f1": f1}\n\n\ndef train_one(name, data):\n    set_seed(CONFIG["seed"])\n    spec = CONFIG["models"][name]\n    directory = OUTPUT / spec["repo"].split("/")[1]\n    tokenizer = AutoTokenizer.from_pretrained(spec["base"])\n    tokenizer.padding_side = "right"\n    if tokenizer.pad_token is None:\n        tokenizer.pad_token = tokenizer.eos_token\n    model = AutoModelForSequenceClassification.from_pretrained(\n        spec["base"], num_labels=2, id2label=ID2LABEL,\n        label2id={value: key for key, value in ID2LABEL.items()})\n    model.config.pad_token_id = tokenizer.pad_token_id\n    model.config.use_cache = False\n    encoded = data.map(lambda batch: tokenizer(batch["text"], truncation=True,\n                       max_length=CONFIG["max_length"]), batched=True, remove_columns=["text"])\n    args = TrainingArguments(\n        output_dir=str(directory / "checkpoints"), num_train_epochs=CONFIG["epochs"],\n        learning_rate=2e-5, weight_decay=0.01, warmup_ratio=0.1,\n        per_device_train_batch_size=8, per_device_eval_batch_size=16,\n        gradient_accumulation_steps=2, eval_strategy="epoch", save_strategy="epoch",\n        load_best_model_at_end=True, metric_for_best_model="macro_f1",\n        greater_is_better=True, save_total_limit=1, logging_steps=50,\n        report_to="none", seed=CONFIG["seed"], data_seed=CONFIG["seed"],\n        fp16=torch.cuda.is_available(), dataloader_num_workers=0)\n    trainer = Trainer(model=model, args=args, train_dataset=encoded["train"],\n                      eval_dataset=encoded["validation"], processing_class=tokenizer,\n                      data_collator=DataCollatorWithPadding(tokenizer), compute_metrics=compute_metrics)\n    started = time.perf_counter()\n    trainer.train()\n    training_seconds = time.perf_counter() - started\n    # The test split is evaluated only after validation selects the checkpoint.\n    predictions = trainer.predict(encoded["test"], metric_key_prefix="test")\n    metrics = {key: float(value) for key, value in predictions.metrics.items()}\n    metrics["training_seconds"] = training_seconds\n    trainer.save_model(str(directory / "model"))\n    tokenizer.save_pretrained(directory / "model")\n    predicted = predictions.predictions.argmax(axis=-1)\n    matrix = confusion_matrix(predictions.label_ids, predicted, labels=[0, 1]).tolist()\n    (directory / "model" / "metrics.json").write_text(json.dumps(metrics, indent=2))\n    (directory / "model" / "confusion_matrix.json").write_text(json.dumps({"labels": list(ID2LABEL.values()), "rows": "true", "columns": "predicted", "matrix": matrix}, indent=2))\n    (directory / "model" / "experiment_config.json").write_text(json.dumps(CONFIG, indent=2))\n    pd.DataFrame({"true_label": predictions.label_ids, "predicted_label": predicted}).to_csv(directory / "test_predictions.csv", index=False)\n    card = f\'\'\'---\nlanguage: en\nlicense: apache-2.0\nlibrary_name: transformers\npipeline_tag: text-classification\nbase_model: {spec["base"]}\ndatasets:\n- stanfordnlp/imdb\n---\n# {name}: IMDb sentiment\n\nFine-tuned for Sultan0336\'s encoder/decoder comparison assignment.\nBase model: `{spec["base"]}`. Two-class sequence classification: 0 NEGATIVE, 1 POSITIVE.\nDistilBERT is bidirectional; DistilGPT2 uses causal attention with a classification head.\nThis checkpoint predicts sentiment; it is not a conversational generator.\n\n## Data and training\nIMDb: 4,000 training reviews, 500 validation reviews, and 1,000 held-out test reviews.\nEach split is class-balanced. Validation comes from the original training split;\ntest comes from the original test split. Exact normalized duplicates are removed.\nHTML line breaks are removed and whitespace is normalized. Maximum length: 128 tokens.\nSeed 42, 2 epochs, AdamW, learning rate 0.00002, weight decay 0.01,\nwarmup ratio 0.1, batch size 8, accumulation 2. Best validation macro-F1 selects the checkpoint.\n\n## Measured held-out results\n```json\n{json.dumps(metrics, indent=2)}\n```\n\n## Limitations\nSmall training sample; one seed; movie-review domain; no neutral label.\nLong reviews are truncated. Sarcasm and mixed sentiment can be misclassified.\nSoftmax scores are not calibrated probabilities. This comparison cannot isolate architecture\nfrom pretraining, tokenizer, and parameter-count differences.\nDataset terms: https://huggingface.co/datasets/stanfordnlp/imdb\n\n## Load\n```python\nfrom transformers import pipeline\nclassifier = pipeline("text-classification", model="{spec["repo"]}")\nprint(classifier("An enjoyable movie.", truncation=True, max_length=128))\n```\n\'\'\'\n    (directory / "model" / "README.md").write_text(card, encoding="utf-8")\n    del trainer, model, tokenizer, encoded, predictions\n    gc.collect()\n    if torch.cuda.is_available():\n        torch.cuda.empty_cache()\n    return {"model": name, **metrics}\n\n\ndef upload_and_verify():\n    api = HfApi()\n    assert api.whoami()["name"].casefold() == CONFIG["hf_username"].casefold(), "Log in as Sultan0336."\n    verified = {}\n    for name, spec in CONFIG["models"].items():\n        folder = OUTPUT / spec["repo"].split("/")[1] / "model"\n        assert (folder / "metrics.json").exists(), "Train both models first."\n        api.create_repo(repo_id=spec["repo"], private=False, exist_ok=True)\n        commit = api.upload_folder(repo_id=spec["repo"], folder_path=str(folder))\n        tokenizer = AutoTokenizer.from_pretrained(spec["repo"], revision=commit.oid, force_download=True)\n        model = AutoModelForSequenceClassification.from_pretrained(spec["repo"], revision=commit.oid, force_download=True)\n        model.eval()\n        with torch.inference_mode():\n            logits = model(**tokenizer("An enjoyable movie.", return_tensors="pt")).logits\n        assert logits.shape == (1, 2) and torch.isfinite(logits).all()\n        verified[name] = {"repo": spec["repo"], "commit": commit.oid, "remote_reload": "passed"}\n        del tokenizer, model\n        gc.collect()\n    (OUTPUT / "upload_verification.json").write_text(json.dumps(verified, indent=2))\n    return verified\n'}
for name, source in SOURCE_FILES.items():
    Path(name).write_text(source, encoding="utf-8")
print("Project files ready.")


## 3. Prepare the data
Download IMDb and create balanced splits: 4,000 training, 500 validation,
1,000 test reviews. Exact normalized duplicates are removed, including overlaps
between the original train and test sets. Test data never selects the checkpoint.
The saved split hashes record exactly which cleaned texts were used.


In [ ]:
from train_models import CONFIG, OUTPUT, prepare_data, train_one, upload_and_verify
data = prepare_data()
print(data)
for name, split in data.items():
    print(name, {label: list(split["label"]).count(label) for label in (0, 1)})


## 4. Fine-tune the encoder
Two epochs, learning rate 2e-5, effective batch size 16, maximum 128 tokens.
The randomly initialized classifier warning is expected. The pretrained backbone
and new head are both trained. The best validation macro-F1 checkpoint is saved.


In [ ]:
encoder_result = train_one("Encoder: DistilBERT", data)
print(encoder_result)


## 5. Fine-tune the decoder
Use the same splits and settings. Its tokenizer uses right padding and the EOS token as padding.

In [ ]:
decoder_result = train_one("Decoder: DistilGPT2", data)
print(decoder_result)


## 6. Compare actual held-out results and create a report
Read the table before drawing conclusions. Confusion matrices have true labels as rows and predicted labels as columns.

In [ ]:
import pandas as pd
import shutil
import subprocess
results = Path("results")
results.mkdir(exist_ok=True)
comparison = pd.DataFrame([encoder_result, decoder_result])
comparison.to_csv(results / "comparison.csv", index=False)
display(comparison)
shutil.copy(OUTPUT / "split_manifest.json", results / "split_manifest.json")
shutil.copy("project_config.json", results / "experiment_config.json")
(results / "environment.txt").write_text(subprocess.check_output(["python", "-m", "pip", "freeze"], text=True))
for spec in CONFIG["models"].values():
    slug = spec["repo"].split("/")[1]
    shutil.copy(OUTPUT / slug / "model" / "confusion_matrix.json", results / f"{slug}_confusion_matrix.json")
    shutil.copy(OUTPUT / slug / "test_predictions.csv", results / f"{slug}_test_predictions.csv")
    print(slug, (results / f"{slug}_confusion_matrix.json").read_text())
lines = ["# Encoder and Decoder Sentiment Comparison", "",
         "## Method", "We fine-tuned DistilBERT and DistilGPT2 on the same balanced IMDb subset: 4,000 training, 500 validation and 1,000 held-out test reviews. Both backbones and classification heads were updated. The best validation macro-F1 checkpoint was selected for each model. Test data was used only after checkpoint selection.", "",
         "Settings: seed 42, two epochs, learning rate 2e-5, effective batch size 16, maximum 128 tokens. Exact normalized duplicates were removed before sampling.", "",
         "## Actual test results", "| Model | Accuracy | Macro precision | Macro recall | Macro F1 |", "|---|---:|---:|---:|---:|"]
for row in [encoder_result, decoder_result]:
    lines.append(f"| {row['model']} | {row['test_accuracy']:.4f} | {row['test_macro_precision']:.4f} | {row['test_macro_recall']:.4f} | {row['test_macro_f1']:.4f} |")
gap = encoder_result["test_macro_f1"] - decoder_result["test_macro_f1"]
finding = "The models tied on test macro-F1." if abs(gap) < 1e-12 else f"{'DistilBERT' if gap > 0 else 'DistilGPT2'} had higher test macro-F1 by {abs(gap):.4f} in this run."
lines += ["", "## Interpretation", finding, "This one run does not establish that either architecture is universally better. Tokenizers, pretraining and model size differ. Movie-review results may not transfer to other domains. Long reviews are truncated and neutral sentiment is unsupported.", "", "## Deployment evidence", "Hugging Face upload verification and live Streamlit checks must be added after they succeed. Record the public app URL and examples you actually tested.", "", "## Sources", "- https://huggingface.co/datasets/stanfordnlp/imdb", "- https://huggingface.co/distilbert/distilbert-base-uncased", "- https://huggingface.co/distilbert/distilgpt2"]
(results / "report.md").write_text("\n".join(lines), encoding="utf-8")


## 7. Sign in to Hugging Face
Use the private login widget with a write token belonging to **Sultan0336**.
Never save your token in a notebook cell. The next cell publishes two public model repositories.


In [ ]:
from huggingface_hub import notebook_login
notebook_login()


## 8. Upload both models and verify remote loading
This can take time. Each uploaded revision is downloaded again and checked for valid predictions.

In [ ]:
verification = upload_and_verify()
print(json.dumps(verification, indent=2))
shutil.copy(OUTPUT / "upload_verification.json", results / "upload_verification.json")


## 9. Download results
Extract this ZIP and upload the results folder to GitHub beside app.py. Keep the notebook outputs as evidence.

In [ ]:
from google.colab import files
shutil.make_archive("assignment_results", "zip", root_dir=".", base_dir="results")
files.download("assignment_results.zip")


## Optional: download a model backup
The backup is large. Your verified Hugging Face repositories also store the final models. Colab local storage disappears when the runtime is deleted.

In [ ]:
shutil.make_archive("trained_models_backup", "zip", root_dir=".", base_dir="outputs")
files.download("trained_models_backup.zip")


## 10. Deploy and verify
Follow README.md to upload app.py, text_utils.py, requirements.txt,
project_config.json and results/ to GitHub. Deploy app.py on Streamlit Community
Cloud with Python 3.11. Test each model and Compare both, then open the app in a
private browser window. Submit the actual public URL. A successful notebook
alone does not prove the deployed app works.
